# Assignment 1 — Portfolio Optimization
## Strait of Hormuz Disruption: Second-Order Commodity Plays

**Methodology overview:**
1. Pull 3 years of daily returns (2022–2024) for all 17 stocks
2. Collapse each bucket to a single synthetic return series using fixed within-bucket weights
3. Monte Carlo over 10,000 bucket-weight draws to map the efficient frontier
4. Select the max-Sharpe portfolio and produce final per-stock weights for QuantConnect

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import yfinance as yf

plt.style.use('seaborn-v0_8-whitegrid')
SEED = 42
rng = np.random.default_rng(SEED)

## 1. Portfolio Definition

Five thematic buckets. Within-bucket weights are set by conviction ahead of optimization.

In [ ]:
BUCKETS = {
    'Aluminum': {
        'AA':    0.50,   # Alcoa — hydro-powered, captures full revenue upside
        'CENX':  0.25,   # Century Aluminum — highest beta to Al price
        'NHYDY': 0.25,   # Norsk Hydro ADR — fully hydro-powered, geopolitically neutral
    },
    'LNG_Gas': {
        'LNG':   0.25,   # Cheniere Energy — largest US LNG exporter
        'EQT':   0.25,   # EQT — largest US natural gas producer
        'AR':    0.25,   # Antero Resources — high LNG export leverage
        'GLNG':  0.25,   # Golar LNG — shipping + FLNG assets
    },
    'Fertilizer': {
        'CF':    1/3,    # CF Industries — largest US nitrogen producer
        'MOS':   1/3,    # Mosaic — US potash + phosphate
        'IPI':   1/3,    # Intrepid Potash — only significant US domestic potash pure-play
    },
    'Defense': {
        'LMT':   0.60,   # Lockheed Martin — THAAD + PAC-3 interceptors (the consumable)
        'RTX':   0.30,   # RTX — SM-3, Patriot system
        'NOC':   0.10,   # Northrop — IBCS battle network integration
    },
    'Tankers': {
        'FRO':   1/3,    # Frontline — VLCC crude tankers
        'STNG':  1/3,    # Scorpio Tankers — product tankers
        'INSW':  1/3,    # International Seaways — mixed fleet
    },
}

ALL_TICKERS = [t for bucket in BUCKETS.values() for t in bucket]
print(f"{len(ALL_TICKERS)} tickers across {len(BUCKETS)} buckets")
print(ALL_TICKERS)

## 2. Pull Price Data (2022–2024)

Optimization window: Jan 2022 – Dec 2024. Backtest in QuantConnect will run 2025–present (out-of-sample).

In [ ]:
START = '2022-01-01'
END   = '2024-12-31'

raw = yf.download(ALL_TICKERS, start=START, end=END, auto_adjust=True, progress=False)['Close']

print(f"Downloaded: {raw.shape[0]} trading days, {raw.shape[1]} tickers")
print(f"Date range: {raw.index[0].date()} → {raw.index[-1].date()}")
print("\nMissing data per ticker:")
print(raw.isnull().sum().to_string())

In [ ]:
returns = raw.pct_change().dropna(how='all')

# Forward-fill any gaps (e.g. ADR non-trading days), then drop any remaining NaN rows
returns = returns.ffill().dropna()

print(f"Returns shape after cleaning: {returns.shape}")
print(f"\nAnnualized return (simple mean × 252):")
ann_ret = (returns.mean() * 252).sort_values(ascending=False)
print(ann_ret.apply(lambda x: f"{x:.1%}").to_string())

## 3. Construct Bucket Return Series

Each bucket collapses to a single synthetic daily return using the conviction-weighted within-bucket weights. This is what the Monte Carlo optimizes over.

In [ ]:
bucket_returns = pd.DataFrame(index=returns.index)

for bucket_name, weights in BUCKETS.items():
    tickers = list(weights.keys())
    w = np.array(list(weights.values()))
    w = w / w.sum()  # normalise to be safe
    bucket_returns[bucket_name] = (returns[tickers] * w).sum(axis=1)

print("Bucket annualised returns (2022–2024):")
bucket_ann = (bucket_returns.mean() * 252)
bucket_vol = (bucket_returns.std() * np.sqrt(252))
bucket_sr  = bucket_ann / bucket_vol  # rough Sharpe (ignoring rf here for display)

summary = pd.DataFrame({'Ann. Return': bucket_ann, 'Ann. Vol': bucket_vol, 'Sharpe (rf=0)': bucket_sr})
print(summary.applymap(lambda x: f"{x:.3f}").to_string())

print("\nBucket return correlation matrix:")
print(bucket_returns.corr().applymap(lambda x: f"{x:.2f}").to_string())

## 4. Monte Carlo Optimization

10,000 random weight vectors drawn from a Dirichlet distribution (uniform over the simplex).  
Constraints are **asymmetric by bucket** to encode strategic intent:

| Bucket | Min | Max | Rationale |
|---|---|---|---|
| Aluminum | 10% | 40% | Core thesis — structurally insulated from strait resolution |
| LNG/Gas | 10% | 40% | Core thesis — moderate resolution lag |
| Fertilizer | 10% | 40% | Core thesis — domestic potash is strait-independent |
| Defense | 10% | 40% | Structurally insulated — procurement cycles multi-year regardless |
| Tankers | 5% | 20% | **Tactical only** — capped because this is the one bucket we'd exit on resolution |

Objective: maximise Sharpe ratio.

In [ ]:
# Risk-free rate: approximate current 3-month T-bill (annualised)
RF_RATE = 0.043  # ~4.3% as of late 2024; adjust if needed

N_SIM        = 10_000
N_BUCKETS    = len(BUCKETS)
BUCKET_NAMES = list(BUCKETS.keys())

# Per-bucket [min, max] constraints — asymmetric to reflect strategic intent
BUCKET_BOUNDS = {
    'Aluminum':   (0.10, 0.40),
    'LNG_Gas':    (0.10, 0.40),
    'Fertilizer': (0.10, 0.40),
    'Defense':    (0.10, 0.40),
    'Tankers':    (0.05, 0.20),  # tactical cap
}
W_MINS = np.array([BUCKET_BOUNDS[b][0] for b in BUCKET_NAMES])
W_MAXS = np.array([BUCKET_BOUNDS[b][1] for b in BUCKET_NAMES])

# Pre-compute bucket covariance matrix (annualised)
cov_matrix   = bucket_returns.cov() * 252
mean_returns = bucket_returns.mean() * 252

def sample_constrained_weights(n_target, w_mins, w_maxs, rng, batch=50_000):
    """
    Rejection sampler: draw from Dirichlet, clip to per-bucket bounds,
    renormalise, keep only samples that still satisfy all bounds after renorm.
    Repeats in batches until n_target valid samples are collected.
    """
    n_assets = len(w_mins)
    collected = []
    while sum(len(c) for c in collected) < n_target:
        raw     = rng.dirichlet(np.ones(n_assets), size=batch)
        clipped = np.clip(raw, w_mins, w_maxs)
        normed  = clipped / clipped.sum(axis=1, keepdims=True)
        valid   = np.all((normed >= w_mins) & (normed <= w_maxs), axis=1)
        collected.append(normed[valid])
    return np.vstack(collected)[:n_target]

weights_matrix = sample_constrained_weights(N_SIM, W_MINS, W_MAXS, rng)

# Vectorised portfolio metrics
port_returns = weights_matrix @ mean_returns.values
port_vols    = np.sqrt(
    np.einsum('ij,jk,ik->i', weights_matrix, cov_matrix.values, weights_matrix)
)
port_sharpes = (port_returns - RF_RATE) / port_vols

print(f"Simulated {N_SIM:,} portfolios")
print(f"Return range:  {port_returns.min():.1%} – {port_returns.max():.1%}")
print(f"Vol range:     {port_vols.min():.1%} – {port_vols.max():.1%}")
print(f"Sharpe range:  {port_sharpes.min():.3f} – {port_sharpes.max():.3f}")

## 5. Efficient Frontier Plot

In [ ]:
best_idx    = np.argmax(port_sharpes)
minvol_idx  = np.argmin(port_vols)

fig, ax = plt.subplots(figsize=(10, 6))

sc = ax.scatter(
    port_vols, port_returns,
    c=port_sharpes, cmap='plasma', alpha=0.4, s=8, linewidths=0
)
plt.colorbar(sc, ax=ax, label='Sharpe Ratio')

# Max Sharpe
ax.scatter(
    port_vols[best_idx], port_returns[best_idx],
    marker='*', s=300, color='gold', edgecolors='black', linewidths=0.8, zorder=5,
    label=f'Max Sharpe  ({port_sharpes[best_idx]:.3f})'
)
# Min Vol
ax.scatter(
    port_vols[minvol_idx], port_returns[minvol_idx],
    marker='D', s=120, color='cyan', edgecolors='black', linewidths=0.8, zorder=5,
    label=f'Min Vol  ({port_vols[minvol_idx]:.1%})'
)

ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.set_xlabel('Annualised Volatility', fontsize=12)
ax.set_ylabel('Annualised Return', fontsize=12)
ax.set_title('Efficient Frontier — 5-Bucket Monte Carlo (10,000 simulations)', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('efficient_frontier.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: efficient_frontier.png")

## 6. Optimal Weights

In [ ]:
optimal_bucket_weights = dict(zip(BUCKET_NAMES, weights_matrix[best_idx]))

print("=" * 50)
print("MAX SHARPE PORTFOLIO")
print("=" * 50)
print(f"  Annualised Return : {port_returns[best_idx]:.2%}")
print(f"  Annualised Vol    : {port_vols[best_idx]:.2%}")
print(f"  Sharpe Ratio      : {port_sharpes[best_idx]:.3f}")
print()
print("Bucket weights:")
for bucket, w in optimal_bucket_weights.items():
    print(f"  {bucket:<15} {w:.1%}")

print()
print("MIN VOL PORTFOLIO (for comparison)")
print("=" * 50)
print(f"  Annualised Return : {port_returns[minvol_idx]:.2%}")
print(f"  Annualised Vol    : {port_vols[minvol_idx]:.2%}")
print(f"  Sharpe Ratio      : {port_sharpes[minvol_idx]:.3f}")
print()
print("Bucket weights:")
for bucket, w in zip(BUCKET_NAMES, weights_matrix[minvol_idx]):
    print(f"  {bucket:<15} {w:.1%}")

## 7. Final Per-Stock Weights for QuantConnect

Multiply each bucket's optimal weight by its within-bucket weights to get final allocations.

In [ ]:
final_weights = {}
for bucket_name, stocks in BUCKETS.items():
    bucket_w = optimal_bucket_weights[bucket_name]
    w_arr = np.array(list(stocks.values()))
    w_arr = w_arr / w_arr.sum()
    for ticker, within_w in zip(stocks.keys(), w_arr):
        final_weights[ticker] = bucket_w * within_w

# Sanity check
total = sum(final_weights.values())
assert abs(total - 1.0) < 1e-9, f"Weights don't sum to 1: {total}"

print("Final per-stock weights (copy into QuantConnect):")
print()
print("HOLDINGS = {")
for bucket_name, stocks in BUCKETS.items():
    print(f"    # --- {bucket_name} ({optimal_bucket_weights[bucket_name]:.1%} of portfolio) ---")
    for ticker in stocks:
        print(f"    '{ticker}': {final_weights[ticker]:.6f},")
print("}")
print()
print(f"Total weight: {total:.8f}")

## 8. Bucket Weight Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: bucket-level pie
bucket_labels = [f"{k}\n{v:.1%}" for k, v in optimal_bucket_weights.items()]
colors = ['#2196F3', '#FF9800', '#4CAF50', '#9C27B0', '#F44336']
axes[0].pie(
    list(optimal_bucket_weights.values()),
    labels=bucket_labels,
    colors=colors,
    autopct='',
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
axes[0].set_title('Optimal Bucket Allocation (Max Sharpe)', fontsize=12)

# Right: per-stock bar chart
tickers_sorted = sorted(final_weights, key=final_weights.get, reverse=True)
vals = [final_weights[t] for t in tickers_sorted]

# Colour bars by bucket
ticker_to_bucket = {t: b for b, stocks in BUCKETS.items() for t in stocks}
bucket_to_color = dict(zip(BUCKETS.keys(), colors))
bar_colors = [bucket_to_color[ticker_to_bucket[t]] for t in tickers_sorted]

bars = axes[1].bar(tickers_sorted, [v * 100 for v in vals], color=bar_colors, edgecolor='white')
axes[1].set_ylabel('Portfolio Weight (%)', fontsize=11)
axes[1].set_title('Per-Stock Allocation', fontsize=12)
axes[1].tick_params(axis='x', rotation=45)

# Legend for buckets
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=bucket_to_color[b], label=b) for b in BUCKETS]
axes[1].legend(handles=legend_elements, fontsize=9, loc='upper right')

plt.tight_layout()
plt.savefig('portfolio_weights.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: portfolio_weights.png")

---\n## 9. Thesis Validation: Performance Since Strait Closure\n\nIran declared the Strait of Hormuz closed on **February 28, 2026**. This section downloads price data from that date to present and shows how each bucket has performed since the thesis became active.\n\nThis is *not* in-sample — none of this data was used in the optimization.

In [ ]:
CLOSURE_DATE = '2026-02-28'
TODAY        = pd.Timestamp.today().strftime('%Y-%m-%d')

raw_post = yf.download(ALL_TICKERS, start=CLOSURE_DATE, end=TODAY, auto_adjust=True, progress=False)['Close']
ret_post = raw_post.pct_change().ffill().dropna()

print(f"Post-closure data: {ret_post.shape[0]} trading days")
print(f"Period: {ret_post.index[0].date()} → {ret_post.index[-1].date()}")

# Construct bucket return series over the post-closure period
bucket_ret_post = pd.DataFrame(index=ret_post.index)
for bucket_name, weights in BUCKETS.items():
    tickers = list(weights.keys())
    w = np.array(list(weights.values())); w /= w.sum()
    bucket_ret_post[bucket_name] = (ret_post[tickers] * w).sum(axis=1)

# Compute cumulative returns (rebased to 1.0 at closure date)
cumret_post = (1 + bucket_ret_post).cumprod()

# Also compute the optimally-weighted portfolio cumulative return
opt_bw = np.array([optimal_bucket_weights[b] for b in BUCKET_NAMES])
port_daily_post = bucket_ret_post.values @ opt_bw
port_cumret_post = (1 + port_daily_post).cumprod()

# Summary stats since closure
days = len(ret_post)
print(f"\nBucket performance since strait closure ({days} trading days):")
for bucket in BUCKET_NAMES:
    total_ret = cumret_post[bucket].iloc[-1] - 1
    ann_ret   = (1 + total_ret) ** (252 / days) - 1
    print(f"  {bucket:<15}  Total: {total_ret:+.1%}   Annualised: {ann_ret:+.1%}")

port_total = port_cumret_post[-1] - 1
port_ann   = (1 + port_total) ** (252 / days) - 1
print(f"\n  {'PORTFOLIO':<15}  Total: {port_total:+.1%}   Annualised: {port_ann:+.1%}")

In [ ]:
colors = ['#2196F3', '#FF9800', '#4CAF50', '#9C27B0', '#F44336']
bucket_to_color = dict(zip(BUCKET_NAMES, colors))

# Pull SPY as benchmark, align to same trading days as our portfolio
spy_raw   = yf.download('SPY', start=CLOSURE_DATE, end=TODAY, auto_adjust=True, progress=False)['Close'].squeeze()
spy_ret   = spy_raw.pct_change().dropna()
spy_cumret = (1 + spy_ret).cumprod().reindex(cumret_post.index).ffill()
spy_pct   = (spy_cumret - 1) * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: cumulative return lines per bucket + portfolio + SPY benchmark
ax = axes[0]
for bucket, color in bucket_to_color.items():
    ax.plot(cumret_post.index, (cumret_post[bucket] - 1) * 100,
            label=bucket, color=color, linewidth=1.8, alpha=0.85)
ax.plot(cumret_post.index, (port_cumret_post - 1) * 100,
        label='Portfolio', color='black', linewidth=2.5, linestyle='--')
ax.plot(spy_pct.index, spy_pct.values,
        label='SPY (benchmark)', color='gray', linewidth=1.8, linestyle='-.')
ax.axhline(0, color='lightgray', linewidth=0.8, linestyle=':')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:+.0f}%'))
ax.set_title('Cumulative Return Since Strait Closure\n(Feb 28, 2026 → present)', fontsize=11)
ax.set_xlabel('Date')
ax.set_ylabel('Cumulative Return')
ax.legend(fontsize=9)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

# Right: total return bar chart by bucket + SPY reference line
ax2 = axes[1]
bucket_totals = {b: (cumret_post[b].iloc[-1] - 1) * 100 for b in BUCKET_NAMES}
spy_total = spy_pct.iloc[-1]
sorted_buckets = sorted(bucket_totals, key=bucket_totals.get, reverse=True)
bar_colors = [bucket_to_color[b] for b in sorted_buckets]
bars = ax2.bar(sorted_buckets, [bucket_totals[b] for b in sorted_buckets],
               color=bar_colors, edgecolor='white', width=0.6)
ax2.axhline(spy_total, color='gray', linewidth=1.5, linestyle='-.', label=f'SPY: {spy_total:+.1f}%')
ax2.axhline(0, color='lightgray', linewidth=0.8, linestyle=':')
for bar, bucket in zip(bars, sorted_buckets):
    val = bucket_totals[bucket]
    ax2.text(bar.get_x() + bar.get_width() / 2,
             val + (1.5 if val >= 0 else -2.5),
             f'{val:+.1f}%', ha='center', va='bottom' if val >= 0 else 'top',
             fontsize=10, fontweight='bold')
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:+.0f}%'))
ax2.set_title('Total Return by Bucket Since Closure', fontsize=11)
ax2.set_ylabel('Total Return (%)')
ax2.tick_params(axis='x', rotation=20)
ax2.legend(fontsize=9)

plt.suptitle('Thesis Validation: Strait of Hormuz Closure (Feb 28, 2026)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('thesis_validation.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: thesis_validation.png  |  SPY benchmark: {spy_total:+.1f}%")